# IFEval on ScreamingFace: stable first run, then the research experiment

IFEval (arXiv:2311.07911) is 541 prompts with machine-checkable constraints — word
counts, forbidden punctuation, required sections. The Engine grades every response with
a deterministic verifier: **no judge model in the grading path, zero grading cost**.

There is one IFEval family with three explicit Variants:

- `ifeval` — one shot. A solo Model answers once; a Fusion's members answer and its
  synthesizer **blends** them into one new answer. The blend is checked.
- `ifeval/self-corrective` — up to three attempts. The whole Candidate reads the
  checker's violations, **writes its own feedback, and retries**.
- `ifeval/verifying-ensemble` — the current verifying ensemble implementation based on
  Skurikhin et al. (https://openreview.net/forum?id=XSIYfTm2h7): every direct Fusion
  member is checked individually, and the **synthesizer acts as JUDGE** — it picks a
  passing answer word-for-word, or turns the violations into coaching when nobody
  passed. It never writes the answer on this exam.

One rule to remember: **the synthesizer plays two roles.** Blender on `ifeval`,
judge on `ifeval/verifying-ensemble`.

The required cells below use Haiku and Gemini Flash for a provider-stable one-Case
validation. Khoa's Kimi K3 configuration remains in the optional appendix because a
reasoning model can consume its completion budget before emitting answer text, and an
upstream provider can fail even when Gateway discovery succeeds.

## Before running

AI Gateway on `127.0.0.1:9105`, Engine on `127.0.0.1:9108`. From `packages/screamingface/`:

```bash
just stack-prepare   # once — downloads the pinned benchmark cases
just stack-up        # gateway :9105 + engine :9108 (logs: just stack-logs)
```

If `just` is not installed, use two terminals after preparing the assets:

```bash
# Terminal 1
cd ../../apps/aigateway
./run-dev-gateway.sh

# Terminal 2 — start only after Gateway is healthy
cd ../../apps/url4-cloud
URL4_BENCHMARK_ASSETS=/tmp/screamingface-benchmark-assets   uv run url4-cloud serve --local
```

After pulling or merging SDK code, **restart this notebook's kernel before Run All**.
Python keeps already-imported SDK modules in memory; a stale kernel can ask the new Engine
for a pre-merge Benchmark id and receive `unknown_benchmark`.

In [1]:
import screamingface as sf

In [2]:
sf.connect()

PanelWidget(children=(HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efe…

## Stable smoke Candidates

These four cells are a **paid one-Case validation**, not a scientific result. Haiku is the
solo Candidate. The Fusion pairs Haiku with Gemini Flash and uses Flash as its synthesizer,
so the synthesizer is also a direct member — the shape used by Skurikhin et al. ([Ens-1]).

`progress=True` shows the live Engine stream. Raw URL4 node names are expected until
semantic Case/attempt events land.

In [3]:
smoke_model = sf.Model(
    "openrouter/anthropic/claude-haiku-4.5",
    params={"max_tokens": 4096},
)
smoke_judge = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    params={"max_tokens": 4096},
)

smoke_fusion = sf.Fusion(
    [smoke_model, smoke_judge],
    name="haiku-flash-smoke",
    synthesizer="openrouter/google/gemini-3-flash-preview",
    params={"max_tokens": 4096},
)
smoke_fusion

Fusion(['claude-haiku-4.5', 'gemini-3-flash-preview'], name='haiku-flash-smoke', synthesizer='openrouter/google/gemini-3-flash-preview', params={'max_tokens': 4096})

## ① Baseline — one model, one shot

This validates the canonical paper-comparable protocol. One Case is only a plumbing check;
increase `limit` deliberately for a reported score.

In [4]:
canonical_smoke_model = sf.evaluate(
    smoke_model,
    benchmark="ifeval",
    limit=1,
    progress=True,
)
canonical_smoke_model

ScreamingFace · Run started
ScreamingFace · TextNode: TextNode
ScreamingFace · RelUrlNode: RelUrlNode
ScreamingFace · TextNode: TextNode
ScreamingFace · TextNode: TextNode
ScreamingFace · TextNode: TextNode
ScreamingFace · RelUrlNode: RelUrlNode
ScreamingFace · ProcessNode: ProcessNode
ScreamingFace · LazyExprNode: (candidate_result:0.0:/candidate?web_search=false&q=($item.input)!'$candidate')!'$candidate_result'
ScreamingFace · RelUrlNode: RelUrlNode
ScreamingFace · ProcessNode: ProcessNode
ScreamingFace · ProcessNode: ProcessNode
ScreamingFace · MapNode: checked:1.0:(record:0.0:/benchmarks/ifeval/3aa4d787c3ab3d29/check((candidate_result:0.0:/candidate?web_search=false&q=($item.input)!'$candidate')!'$candidate_result')!'$item.id')!'$record'
ScreamingFace · CollectNode: CollectNode
ScreamingFace · ProcessNode: ProcessNode
ScreamingFace · TextNode: TextNode
ScreamingFace · TextNode: TextNode
ScreamingFace · RelUrlNode: RelUrlNode
ScreamingFace · ProcessNode: ProcessNode
ScreamingFace · 

Report(benchmark='ifeval', candidates=['claude-haiku-4.5'], ok=True)

## ② Does blending preserve instructions?

The synthesizer writes one NEW answer from the members' answers — new text the checker
never saw. A blend can break a constraint every member satisfied (add a comma, drop a
section). This cell measures that risk.

In [ ]:
canonical_smoke_fusion = sf.evaluate(
    smoke_fusion,
    benchmark="ifeval",
    limit=1,
    progress=True,
)
canonical_smoke_fusion

## ③ Can a model correct itself?

The ablation the paper never ran: {solo + feedback loop}. The model answers, the
checker reports violations, the model writes its own feedback and retries — up to
three attempts, earliest pass wins.

Cost: five model calls per case (three answers + two self-feedback authorings), all
unrolled.

In [ ]:
corrective_smoke_model = sf.evaluate(
    smoke_model,
    benchmark="ifeval/self-corrective",
    limit=1,
    progress=True,
)
corrective_smoke_model

## ④ The verifying ensemble (the paper's protocol)

Members answer, the checker checks **each draft individually**, and the synthesizer —
acting as judge here — picks a passing answer verbatim, or coaches everyone and retries
when nobody passed. A judge cannot break a constraint a member satisfied, because it
never rewrites the winning text.

Choose a synthesizer that reliably answers tersely: a judge reply that is not a bare
letter gets no vote (the deterministic passers-first rule decides instead), and the
synthesizer inherits provider-default params on this exam.

In [ ]:
corrective_smoke_fusion = sf.evaluate(
    smoke_fusion,
    benchmark="ifeval/verifying-ensemble",
    limit=1,
    progress=True,
)
corrective_smoke_fusion

## Read the smoke results

- ① vs ② — did blending help or hurt instruction-following?
- ① vs ③ — how much does a feedback loop help one model?
- ③ vs ④ — self-correction vs ensemble correction, same loop, same exam.
- ② vs ④ — blend-then-check vs check-then-select.

Cost note: the iterative-correction exam has no early stop yet — all three attempts
always run (and the solo shape adds two self-feedback calls), so its token totals
overstate a stop-on-success system. Compare scores freely within a column; never
compare our costs to the paper's.

With one Case these values prove only that the complete contracts execute. They are not
benchmark results.

In [ ]:
{
    name: {
        "score": report.candidates[0].score,
        "output_tokens": report.usage.output_tokens,
    }
    for name, report in {
        "① ifeval · haiku": canonical_smoke_model,
        "② ifeval · haiku-flash": canonical_smoke_fusion,
        "③ self-corrective · haiku": corrective_smoke_model,
        "④ verifying-ensemble · haiku-flash": corrective_smoke_fusion,
    }.items()
}

## Optional appendix — Khoa's Kimi K3 experiment

This preserves the original research lineup without making it the environment-health
check. It is disabled so **Run All does not spend on it or fail because of an upstream K3
completion**.

Observed failure meanings:

- `case … carried no valid IFEval check record` means a Candidate/provider failed to return
  a scorable answer. The Engine refuses to turn that into a plausible zero score.
- `aigateway returned neither answer content nor tool calls` commonly means a reasoning
  model exhausted its completion budget before emitting final answer text.
- An HTTP 200 from the upstream call does not prove answer content was present.

Set `RUN_KIMI_RESEARCH = True` only after the smoke grid succeeds. Start with one Case;
increase `KIMI_RESEARCH_LIMIT` deliberately. Kimi K3 may need a larger completion budget,
but a higher ceiling cannot repair an upstream provider error.

In [ ]:
RUN_KIMI_RESEARCH = False
KIMI_RESEARCH_LIMIT = 1

kimi = sf.Model("openrouter/moonshotai/kimi-k3", params={"max_tokens": 4096})
haiku = sf.Model("openrouter/anthropic/claude-haiku-4.5")
kimi_fusion = sf.Fusion(
    [kimi, haiku],
    name="kimi-haiku",
    synthesizer="openrouter/moonshotai/kimi-k3",
    params={"max_tokens": 4096},
)

"Enabled" if RUN_KIMI_RESEARCH else "Skipped — set RUN_KIMI_RESEARCH = True to opt in"

In [ ]:
if RUN_KIMI_RESEARCH:
    kimi_canonical = sf.evaluate(
        kimi,
        benchmark="ifeval",
        limit=KIMI_RESEARCH_LIMIT,
        progress=True,
    )
    kimi_blended = sf.evaluate(
        kimi_fusion,
        benchmark="ifeval",
        limit=KIMI_RESEARCH_LIMIT,
        progress=True,
    )
    kimi_self_corrective = sf.evaluate(
        kimi,
        benchmark="ifeval/self-corrective",
        limit=KIMI_RESEARCH_LIMIT,
        progress=True,
    )
    kimi_verifying_ensemble = sf.evaluate(
        kimi_fusion,
        benchmark="ifeval/verifying-ensemble",
        limit=KIMI_RESEARCH_LIMIT,
        progress=True,
    )
    kimi_results = {
        "① ifeval · kimi": kimi_canonical,
        "② ifeval · kimi-haiku": kimi_blended,
        "③ self-corrective · kimi": kimi_self_corrective,
        "④ verifying-ensemble · kimi-haiku": kimi_verifying_ensemble,
    }
else:
    kimi_results = {}

{
    name: {
        "score": report.candidates[0].score,
        "output_tokens": report.usage.output_tokens,
    }
    for name, report in kimi_results.items()
}